<a href="https://colab.research.google.com/github/potat0w/Aaagh-more-math/blob/main/Copy_of_DL_Assignment_03_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DL Assignment 03

**Name:*Kahon binte zaman*

**Course Email:*kahonbintezaman@gmail.com*  


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:
Loading: The dataset was loaded from the URL using Pandas and stored in a DataFrame.

Missing Values: Some columns had 0 values like Glucose or BMI, which are not realistic. I replaced those zeros with the column’s mean value.

Scaling: I applied StandardScaler to normalize the features so that large values do not dominate smaller ones during training.

Tensors: Finally, I converted the data from NumPy arrays to PyTorch tensors, since PyTorch models work with tensors during training.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DPF', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_to_fix:
    df[col] = df[col].replace(0, df[col].mean())

X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print(f"Training shapes: {X_train.shape}, {y_train.shape}")

Training shapes: torch.Size([614, 8]), torch.Size([614, 1])


# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:
1. Hidden Neurons:
The dataset has 8 input features. I used 16 neurons in the first hidden layer and 8 in the second to keep the model simple and avoid overfitting.

2. Activation Functions:
ReLU is used in the hidden layers for better learning. Sigmoid is used in the output layer since this is a binary classification problem.

3. Trainable Parameters:
The model has 289 trainable parameters in total.

In [3]:
import torch
import torch.nn as nn

class DiabetesModel(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        out = self.network(x)
        return out


model = DiabetesModel(8)

print("Model:")
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters:", total_params)

Model:
DiabetesModel(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
    (5): Sigmoid()
  )
)
Trainable parameters: 289


# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:
Loss Function: We use BCELoss because this is binary problem. It check difference between predicted probability and real label 0 or 1

Optimizer: We use Adam. It change weights fast and help model learn quicker and handle noisy data better.

In [7]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.BCELoss()
learning_rate = 0.01
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Loss Function:", criterion)
print("Optimizer:", optimizer)

Loss Function: BCELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)


# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:
Forward Pass: Model predicts output from inputs.

Loss: Measures how wrong predictions are.

Backward Pass: Compute gradients for weights.

Update Weights: Optimizer adjusts weights to reduce loss.

Reset Gradients: Clear old gradients for next step.

Track Loss: Save loss to check progress.

In [8]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100
loss_history = []

for epoch in range(1, epochs + 1):
    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    loss_history.append(loss.item())

    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{epochs} - Loss: {loss.item():.4f}")

Epoch 10/100 - Loss: 0.5543
Epoch 20/100 - Loss: 0.4714
Epoch 30/100 - Loss: 0.4421
Epoch 40/100 - Loss: 0.4254
Epoch 50/100 - Loss: 0.4108
Epoch 60/100 - Loss: 0.3976
Epoch 70/100 - Loss: 0.3871
Epoch 80/100 - Loss: 0.3795
Epoch 90/100 - Loss: 0.3720
Epoch 100/100 - Loss: 0.3642


# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:
Accuracy: Training accuracy is usually slightly higher than testing accuracy.

Compare Performance: If both are close, the model is fitting well.

Overfitting vs Underfitting:

If training is much higher than testing - overfitting.

If both are low - underfitting.

If both are similar and reasonably high - model is good.

In [9]:
with torch.no_grad():

    train_preds = (model(X_train) > 0.5).float()
    train_acc = (train_preds == y_train).float().mean()

    test_preds = (model(X_test) > 0.5).float()
    test_acc = (test_preds == y_test).float().mean()

print(f"Training Accuracy: {train_acc.item()*100:.2f}%")
print(f"Testing Accuracy: {test_acc.item()*100:.2f}%")

Training Accuracy: 83.71%
Testing Accuracy: 75.97%


# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:
I changed learning rate 0.01 to 0.1

Training becomes fast in start

Loss sometimes jump up-down at the end

Big LR is fast but unstable, small LR is slow but safe

In [ ]:
new_model = DiabetesModel(8)
new_optimizer = optim.Adam(new_model.parameters(), lr=0.1)
new_criterion = nn.BCELoss()

epochs = 100
new_loss_history = []

for epoch in range(1, epochs + 1):
    preds = new_model(X_train)
    loss = new_criterion(preds, y_train)

    loss.backward()
    new_optimizer.step()
    new_optimizer.zero_grad()

    new_loss_history.append(loss.item())

    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{epochs} - Loss: {loss.item():.4f}")

print(f"Final Loss with LR 0.01: {loss_history[-1]:.4f}")
print(f"Final Loss with LR 0.1: {new_loss_history[-1]:.4f}")

# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:
Why reset gradients every epoch?


PyTorch adds gradients by default. If we don’t reset them, old gradients mix with new ones - wrong weight updates.

If learning rate is too high:


Steps are too big - loss jumps, training is unstable, model may not learn.

If learning rate is too small:


Steps are very small - training is slow, model may get stuck, takes long to learn.

Why define layers in __init__ and not in forward():


Layers in __init__ are created once and weights are saved.
If we define them in forward(), new random weights are created every time model cannot learn.